# One design on Colab

RFD4-Proteina on a free Colab GPU, with no checkout and no build: `pip install` the published
package, generate one enzyme design, download the result.

Pick a GPU runtime first — **Runtime → Change runtime type → T4/L4/A100 GPU**. The first cell checks
the runtime and stops with a clear message if it cannot work, rather than failing halfway through a
long install. See the [hardware requirements](https://rfd4-proteina.org/latest/installation.html#supported-hardware)
for the GPUs this model is qualified on.

Refolding and experimental validation are separate; see the full
[enzyme design tutorial](enzyme_design_tutorial.ipynb).


In [ ]:
"""Check the runtime before installing anything, so a mismatch fails in seconds."""
import re
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise SystemExit(
        f"RFD4-Proteina requires Python 3.12; this runtime is {sys.version.split()[0]}. "
        "Colab's Python is not selectable, so use the Brev launchable or a local install instead: "
        "https://rfd4-proteina.org/latest/installation.html"
    )

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode != 0:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > GPU, then run this cell again.")

# The driver's CUDA major decides the extra: the two are mutually exclusive, and picking the wrong
# one installs operator wheels that only fail once the model reaches them.
match = re.search(r"CUDA Version:\s*(\d+)", smi.stdout)
if not match:
    raise SystemExit("Could not read the CUDA version from nvidia-smi:\n" + smi.stdout)
CUDA_MAJOR = int(match.group(1))
if CUDA_MAJOR < 12:
    raise SystemExit(f"CUDA {CUDA_MAJOR} driver is too old; RFD4-Proteina needs CUDA 12 or 13.")
EXTRA = "all-cuda13" if CUDA_MAJOR >= 13 else "all-cuda12"

print(f"Python {sys.version.split()[0]}, CUDA {CUDA_MAJOR} driver -> rfd4-proteina[{EXTRA}]")
gpu = re.search(r"\|\s+\d+\s+(.+?)\s+(On|Off)", smi.stdout)  # cosmetic; layout varies by driver
print("GPU:", gpu.group(1).strip() if gpu else "detected")


In [ ]:
"""Install the published package. No clone, no build: one wheel set from PyPI."""
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

# --torch-backend=auto matches the Torch wheel to the driver above, so it agrees with EXTRA.
subprocess.run(
    ["uv", "pip", "install", "--system", "--torch-backend=auto", f"rfd4-proteina[{EXTRA}]"],
    check=True,
)

import shutil

RFD4_GENERATE = shutil.which("rfd4-generate")
assert RFD4_GENERATE, "rfd4-generate is not on PATH; re-run the install cell"
print("Ready:", RFD4_GENERATE)


In [ ]:
"""Generate one design from the MCSA enzyme example."""
import os
import shlex
import tempfile
import urllib.request
from pathlib import Path

NUM_DESIGNS = 1
SEED = 42

WORK = Path("/content/rfd4")
WORK.mkdir(parents=True, exist_ok=True)

# The wheel ships the monomer examples and the configs, not the benchmark inputs, so fetch
# the one enzyme fixture from the public repository.
INPUT = WORK / "M0664_2dhn.cif.zst"
if not INPUT.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/RosettaCommons/RFD4-Proteina/main/"
        "data/benchmarks/mcsa/mcsa_37/M0664_2dhn.cif.zst",
        INPUT,
    )

RUN = Path(tempfile.mkdtemp(prefix="colab-enzyme-", dir=WORK))
env = {**os.environ, "CLUSTER": "local", "DATA_PATH": str(WORK / "data_path")}
command = [
    RFD4_GENERATE,
    f"inputs={INPUT}",
    f"out_dir={RUN}",
    f"num_replicates={NUM_DESIGNS}",
    f"seed={SEED}",
    "batch_size=1",
    "num_workers=0",
    "compile_lengths=null",      # skip the torch.compile warm-up for a single design
    "compress_outputs=false",
    "~metrics",
    "+metrics.inline_metrics={}",  # no metric is instantiated, so none of their inputs are needed
]
print(shlex.join(command), "\nLog:", RUN / "run.log", flush=True)

with (RUN / "run.log").open("w") as log:
    result = subprocess.run(command, env=env, stdout=log, stderr=subprocess.STDOUT)
if result.returncode:
    print("\n".join((RUN / "run.log").read_text().splitlines()[-40:]))
    result.check_returncode()

designs = sorted((RUN / "inference_samples").rglob("*_generated.cif"))
assert len(designs) == NUM_DESIGNS, f"Expected {NUM_DESIGNS} designs, found {len(designs)}; inspect the log"
for path in designs:
    print(path)


In [ ]:
"""Download the run directory — designs, the effective config and the command log."""
import shutil

from google.colab import files

archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
files.download(archive)
